# Eu ai act

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
import re
from langchain_core.documents import Document

In [22]:
def recursive_chunking(md_text, chunk_size, chunk_overlap):
    # preprocessing the text because the text contains the recitals, articles and annex
    # this will search for the articles, annex and recitals
    articles_md = re.search(r'# _Article 1_', md_text)
    annex_md = re.search(r'## _ANNEX I_', md_text)
    if not articles_md or not annex_md:
            raise ValueError("Could not find the start of the Articles or Annexes in the text.")
    start_of_article = articles_md.start()
    start_of_annex = annex_md.start()
    #this will split the text into recitals, articles and annex
    recital_text = md_text[:start_of_article]
    article_text = md_text[start_of_article:start_of_annex]
    annex_text = md_text[start_of_annex:]

    # storing in the langchain document format with metadara
    md = [
        Document(page_content=recital_text, metadata={"section": "recitals"}),
        Document(page_content=article_text, metadata={"section": "articles"}),
        Document(page_content=annex_text, metadata={"section": "annex"})
    ]

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap    )
    return text_splitter.split_documents(md)

In [23]:
md_file_path = "../data/processed/clean_eu_ai_act.md"
with open(md_file_path, "r", encoding="utf-8") as f:
    md_content = f.read()
md_chunks = recursive_chunking(md_content, chunk_size=1000, chunk_overlap=100)
print(f"Code document split into {len(md_chunks)} chunks")

Code document split into 176 chunks


# Eu prohibited ai act

In [25]:
import re
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# 1. Read the messy file
proh_ai_file_path = "../data/processed/clean_eu_proh_ai_act.md"
with open(proh_ai_file_path, "r", encoding="utf-8") as f:
    proh_md_content = f.read()

# --- PREPROCESSING / CLEANING PHASE ---

# Fix 1: Remove fake headers from "For example," boxes
proh_md_content = re.sub(r'# For example,', r'For example,', proh_md_content)

# Fix 2: Remove fake headers from "Article X AI Act provides:" quote boxes
# This targets "# **Article..." or "# **_Article..."
proh_md_content = re.sub(r'# (\*\*_*Article \d+.*\*\*)', r'\1', proh_md_content)

# Fix 3: Upgrade sub-sections (e.g., "# **2.1." -> "## **2.1.")
proh_md_content = re.sub(r'# \*\*(\d+\.\d+\.)', r'## **\1', proh_md_content)

# Fix 4: Upgrade sub-sub-sections (e.g., "# **2.5.1." -> "### **2.5.1.")
proh_md_content = re.sub(r'# \*\*(\d+\.\d+\.\d+\.)', r'### **\1', proh_md_content)

# Fix 5: Upgrade lettered sections (e.g., "# **_a)" -> "#### **_a)")
proh_md_content = re.sub(r'# \*\*_([a-z]\))', r'#### **_\1', proh_md_content)
proh_md_content = re.sub(r'# _([a-z]\))', r'#### _\1', proh_md_content)

# --- LANGCHAIN CHUNKING PHASE ---
    # storing in metadara format

headers_to_split_on = [
    ("#", "Main_Section"),    
    ("##", "Sub_Section"),
    ("###", "Sub_Sub_Section"),
    ("####", "Letter_Section")
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False 
)
# splitting the text into chunks based on the headers defined above
md_header_splits = markdown_splitter.split_text(proh_md_content)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
# applied the recursive chunking to the header splits to create final chunks
final_proh_chunks = text_splitter.split_documents(md_header_splits)

print(f"Success! Created {len(final_proh_chunks)} perfectly structured chunks.")

Success! Created 182 perfectly structured chunks.


# Trying the semantic chunker


In [ ]:
# for the eu ai act


Failed with error: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}}


In [ ]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_google_vertexai import VertexAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
import os
def process_eu_ai_act(md_file_path):
    """
    Processes the EU AI Act by separating it into Recitals, Articles, and Annexes
    before running the recursive character chunking.
    """
    with open(md_file_path, "r", encoding="utf-8") as f:
        md_text = f.read()

    # Chunk the documents while preserving metadata
    semantic_chunker=SemanticChunker()
    semantic_chunks=semantic_chunker.chunk(md_text)
    eu_ai_act_embeddings = VertexAIEmbeddings(
        project="ragbench-aiact",
        model="text-embedding-004",
        #task_type="retrieval_document"
    )
    persist_dir: str = "../framework_chroma_db"
    # Initialize empty Chroma DB
    eu_ai_act_embeddings_vectors = Chroma(
        collection_name="eu_ai_act_semantic_chunks",
        embedding_function=eu_ai_act_embeddings,
        persist_directory=persist_dir
    )
    
    # Add documents in batches to avoid rate limits (100 requests per minute)
    batch_size = 50
    for i in range(0, len(semantic_chunks), batch_size):
        batch = semantic_chunks[i:i + batch_size]
        eu_ai_act_embeddings_vectors.add_documents(batch)
        print(f"Added {min(i + batch_size, len(semantic_chunks))}/{len(semantic_chunks)} chunks for AI Act...")
        if i + batch_size < len(semantic_chunks):
            #time.sleep(32) # Wait 32 seconds before next batch to respect rate limits
            pass
            
    return eu_ai_act_embeddings_vectors

